# EXP07 — Comparação de fichas de revisão

Este notebook faz parte do pacote de reprodução. Leia as explicações antes de executar. **Candidatos não verificados permanecem sem rótulo confirmado.**

## 1. Preparar o ambiente

No Colab, a primeira célula baixa uma cópia do projeto caso ela ainda não esteja disponível. No Python local, use o ambiente e as dependências descritos no README.

In [ ]:
from pathlib import Path
import os, subprocess, sys

def preparar_projeto():
    atual = Path.cwd()
    for pasta in [atual, *atual.parents]:
        if (pasta / "src" / "ethmix" / "__init__.py").exists():
            raiz = pasta
            break
    else:
        raiz = Path("/content/ethereum-mixer-reproducibility")
        if not raiz.exists():
            subprocess.run(["git", "clone", "--depth", "1",
                "https://github.com/BorgesGusta/ethereum-mixer-reproducibility.git",
                str(raiz)], check=True)
    sys.path.insert(0, str(raiz / "src"))
    os.chdir(raiz)
    return raiz

RAIZ = preparar_projeto()
print("Projeto:", RAIZ)


### Bibliotecas

Esta célula instala as bibliotecas necessárias na sessão atual, caso ainda não estejam presentes.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

## 2. Arquivos necessários

Duas fichas CSV: `data/processed/exp07_revisor_a.csv` e `exp07_revisor_b.csv`, com `contrato`, `classificacao` e `fonte`. O programa compara as fichas, mas **não** transforma uma revisão com apoio de IA em avaliação cega independente.

In [ ]:
from ethmix.execucao import abrir_dados
import pandas as pd

pasta = RAIZ / "data" / "processed"
a = abrir_dados(pasta / "exp07_revisor_a.csv", ["contrato", "classificacao", "fonte"])
b = abrir_dados(pasta / "exp07_revisor_b.csv", ["contrato", "classificacao", "fonte"])
comparacao = a.merge(b, on="contrato", suffixes=("_a", "_b"), validate="one_to_one")
comparacao["concordam"] = comparacao.classificacao_a == comparacao.classificacao_b
print("Casos comparados:", len(comparacao))
print("Divergências:", int((~comparacao.concordam).sum()))
display(comparacao.loc[~comparacao.concordam])

## 3. Registrar divergências

O protocolo histórico apontou quatro divergências. A proposta de adjudicação não foi homologada. Portanto, não calculamos kappa nem consideramos esta amostra avaliação independente.

In [ ]:
saida = RAIZ / "results" / "generated" / "EXP07"
saida.mkdir(parents=True, exist_ok=True)
comparacao.to_csv(saida / "comparacao.csv", index=False)
print("Comparação salva em", saida)